# Projeto 2 - Naive Bayes em críticas de filmes do IMDB

## 1. Introdução

**Contexto:** no Projeto 1 escolhemos o Naive Bayes, um algoritmo que classifica um texto contando as palavras de cada classe e vendo qual classe é mais provável. Fizemos um exemplo com avaliações de hotel. Agora vamos aplicar a mesma ideia em dados reais.

**Objetivo:** saber se dá para dizer se alguém gostou de um filme só pelas palavras que usou na crítica.

**Hipóteses do Projeto 1 que vamos testar:**
- H1: críticas negativas que usam palavras positivas (ironia) fazem o modelo errar.
- H2: críticas que citam continuações e remakes (`sequel`, `remake`) são mais negativas.
- H3: críticas negativas usam mais `!` e palavras em MAIÚSCULO que as positivas.

**Dados:** *IMDB Dataset of 50K Movie Reviews* (Kaggle), com duas colunas:
- `review`: texto da crítica, em inglês
- `sentiment`: `positive` ou `negative`

## 2. Análise exploratória

### 2.1 Carregando e limpando os dados

In [ ]:
dados <- read.csv("data/IMDB Dataset.csv")

dim(dados)
sum(is.na(dados))      # valores faltando
sum(duplicated(dados)) # linhas repetidas

Não tem valores faltando, mas tem 418 críticas repetidas. Vou tirar as repetidas e também os `<br />` (pedaços de HTML) que aparecem no meio dos textos.

In [ ]:
dados <- dados[!duplicated(dados), ]
dados$review <- gsub("<br />", " ", dados$review)

table(dados$sentiment)

### 2.2 Análise univariada

Vou criar o número de palavras de cada crítica.

In [ ]:
dados$n_palavras <- sapply(strsplit(dados$review, "\\s+"), length)

summary(dados$n_palavras)

In [ ]:
hist(dados$n_palavras, breaks = 50, main = "Tamanho das críticas",
     xlab = "número de palavras", ylab = "quantidade")

A maioria das críticas tem entre 125 e 280 palavras, mas algumas são bem grandes (até 2459). Por isso a média (229) é maior que a mediana (172).

### 2.3 Análise bivariada

In [ ]:
boxplot(n_palavras ~ sentiment, data = dados, main = "Palavras por sentimento",
        xlab = "sentimento", ylab = "número de palavras")

tapply(dados$n_palavras, dados$sentiment, mean)

### 2.4 O que a EDA mostrou

- As classes continuam balanceadas depois de tirar as repetidas (24698 negativas e 24884 positivas). Chutar ao acaso acerta cerca de 50%.
- O tamanho das críticas é parecido nas duas classes (a média muda uns 4 palavras).
- Os dados estão limpos o suficiente para começar.

## 3. Teste de hipótese

Vou usar nível de significância de 5% em todos os testes.

### 3.1 H2: continuações e remakes

- **H0:** a proporção de críticas negativas é a mesma entre as que citam `sequel` ou `remake` e as que não citam.
- **H1:** é maior entre as que citam.

Como é comparação de duas proporções, uso o teste de proporções.

In [ ]:
dados$negativo <- ifelse(dados$sentiment == "negative", 1, 0)
dados$continuacao <- grepl("\\b(sequel|remake)\\b", tolower(dados$review))

table(dados$continuacao, dados$sentiment)
tapply(dados$negativo, dados$continuacao, mean) # proporção de negativas

In [ ]:
negativas <- tapply(dados$negativo, dados$continuacao, sum)
total <- table(dados$continuacao)

prop.test(c(negativas["TRUE"], negativas["FALSE"]),
          c(total["TRUE"], total["FALSE"]), alternative = "greater")

O p-valor é menor que 0,05 (praticamente zero), então rejeitamos H0. Entre as críticas que citam `sequel` ou `remake`, 62% são negativas, contra 49% nas outras. A hipótese H2 se confirmou.

### 3.2 H3: "!" e palavras em maiúsculo

- **H0:** a média de `!` (e de palavras em maiúsculo) é igual nas críticas negativas e positivas.
- **H1:** a média é maior nas negativas.

Uso o teste t para duas médias. Como são mais de 49 mil críticas, ele funciona bem mesmo com os dados não sendo simétricos. Palavra em maiúsculo é qualquer palavra com 2 ou mais letras todas maiúsculas (isso pega também siglas como TV, mas serve como aproximação).

In [ ]:
contar <- function(padrao, texto) {
  sapply(gregexpr(padrao, texto), function(m) sum(m > 0))
}

dados$n_exclam <- contar("!", dados$review)
dados$n_maiusc <- contar("\\b[A-Z]{2,}\\b", dados$review)

tapply(dados$n_exclam, dados$sentiment, mean)
tapply(dados$n_maiusc, dados$sentiment, mean)

In [ ]:
t.test(n_exclam ~ sentiment, data = dados, alternative = "greater")
t.test(n_maiusc ~ sentiment, data = dados, alternative = "greater")

Os dois p-valores são menores que 0,05, então rejeitamos H0 nos dois casos: as críticas negativas têm mais `!` (1,03 contra 0,93) e mais palavras em maiúsculo (1,80 contra 1,51).

A diferença existe, mas é pequena (menos de 0,3 por crítica). Com tantos dados, até diferenças pequenas ficam significativas. Então H3 se confirma, mas esses sinais sozinhos ajudam pouco a classificar.

A H1 (ironia) não dá para testar só com contas. Vamos olhar ela junto com os erros do modelo.

## 4. Preparação e modelo

O Naive Bayes que fizemos no exemplo do hotel, só que com as críticas:

1. Deixar tudo minúsculo e tirar a pontuação, e separar as palavras.
2. Separar 80% das críticas para treino e 20% para teste.
3. No treino, contar quantas vezes cada palavra aparece nas críticas positivas e nas negativas.
4. Calcular P(palavra | classe) com suavização de Laplace (somar 1 em cada contagem para nenhuma probabilidade ficar zero).
5. Para uma crítica nova, multiplicar as probabilidades das palavras e a probabilidade da classe, e escolher a classe maior. Uso `log` e somo no lugar de multiplicar, porque multiplicar milhares de números pequenos dá zero no computador.

In [ ]:
limpar <- function(texto) {
  strsplit(trimws(gsub("[^a-z' ]", " ", tolower(texto))), "\\s+")
}

set.seed(1)
linhas_treino <- sample(nrow(dados), 0.8 * nrow(dados))
treino <- dados[linhas_treino, ]
teste  <- dados[-linhas_treino, ]

In [ ]:
# contando as palavras de cada classe no treino
palavras_pos <- unlist(limpar(treino$review[treino$negativo == 0]))
palavras_neg <- unlist(limpar(treino$review[treino$negativo == 1]))

vocab <- unique(c(palavras_pos, palavras_neg))
V <- length(vocab) # tamanho do vocabulário
V

In [ ]:
cont_pos <- as.numeric(table(factor(palavras_pos, levels = vocab)))
cont_neg <- as.numeric(table(factor(palavras_neg, levels = vocab)))

# P(palavra | classe) com Laplace, já em log
log_pos <- log((cont_pos + 1) / (sum(cont_pos) + V))
log_neg <- log((cont_neg + 1) / (sum(cont_neg) + V))
names(log_pos) <- vocab
names(log_neg) <- vocab

prior_pos <- mean(treino$negativo == 0)
prior_neg <- 1 - prior_pos

In [ ]:
# devolve 1 se a crítica for negativa e 0 se for positiva
classificar <- function(palavras) {
  palavras <- palavras[palavras %in% vocab]
  nota_pos <- log(prior_pos) + sum(log_pos[palavras])
  nota_neg <- log(prior_neg) + sum(log_neg[palavras])
  ifelse(nota_neg > nota_pos, 1, 0)
}

## 5. Resultados

Rodando o modelo nas críticas de teste (demora uns 40 segundos).

In [ ]:
previsto <- sapply(limpar(teste$review), classificar)

table(real = teste$negativo, previsto = previsto)
mean(previsto == teste$negativo) # acurácia

O modelo acertou **84%** das críticas de teste. Chutar ao acaso daria 50%.

Dos erros, 641 foram críticas negativas que ele achou positivas e 945 foram críticas positivas que ele achou negativas.

Para entender o que o modelo aprendeu, vou ver as palavras que mais puxam para cada lado (só palavras que aparecem pelo menos 500 vezes, para não aparecer nome de filme raro).

In [ ]:
dif <- log_neg - log_pos # positivo = puxa para negativa
freq <- cont_pos + cont_neg
dif <- dif[freq >= 500]

head(sort(dif, decreasing = TRUE), 10) # puxam para negativa
head(sort(dif), 10)                    # puxam para positiva

As palavras fazem sentido: `waste`, `worst`, `awful` e `terrible` puxam para negativa, e `beautifully`, `superb`, `wonderful` e `excellent` puxam para positiva. É o que esperávamos.

**Sobre a H1 (ironia):** o modelo só conta palavras, então não entende ironia nem negação ("not good" conta o `good`). Vendo exemplos de críticas negativas que ele achou positivas:

In [ ]:
erros <- which(teste$negativo == 1 & previsto == 0)
length(erros)
substr(teste$review[erros[1:3]], 1, 300)

Lendo esses exemplos, dá para ver que tem crítica negativa com elogios no meio (elogia o elenco ou o que o filme tentou fazer e depois critica). Como o modelo só soma palavras, esses elogios compensam as palavras negativas e ele erra. Isso apoia a H1, mas não medimos a ironia de forma exata, então não dá para dizer que é só isso.

## 6. Conclusão

**Resumo:** dá para saber se alguém gostou do filme só pelas palavras que usou: o Naive Bayes acertou 84% das críticas de teste, contra 50% de um chute. Sobre as hipóteses:
- H2: confirmada. Críticas que citam `sequel` ou `remake` são mais negativas (62% contra 49%).
- H3: confirmada, mas a diferença é pequena. Negativas têm mais `!` e mais palavras em maiúsculo.
- H1: os erros do modelo sugerem que críticas com elogios e críticas negativas se misturam, mas não medimos a ironia.

**Recomendações:**
1. Para classificar textos de opinião, o Naive Bayes é uma boa primeira tentativa: é simples, rápido e já acerta bastante.
2. Usar `!` e maiúsculas como pistas extras junto com as palavras, em vez de sozinhas.

**Limitações:**
- O modelo ignora a ordem das palavras, então não entende negação nem ironia.
- Só vemos o texto em inglês. O dataset não tem gênero, ano ou país, então a H2 foi checada só por palavras no texto (`sequel`, `remake`), que é uma aproximação.
- Não há críticas neutras (notas 5 e 6 ficaram de fora do dataset).
- Testamos com uma única divisão treino/teste.